# K-Means Clustering from Scratch

**Dataset:** Synthetic blobs + Iris (sklearn built-in)  
**Task:** Unsupervised clustering — discover natural groupings without labels

---

## Overview

**K-Means** partitions $n$ data points into $K$ clusters by minimizing the **within-cluster sum of squares (WCSS)**:

$$\underset{\mathbf{\mu}_1, \ldots, \mathbf{\mu}_K}{\text{minimize}} \sum_{k=1}^{K} \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \mathbf{\mu}_k\|^2$$

### Algorithm (Lloyd's Algorithm)

1. **Initialize** $K$ cluster centroids (randomly or via k-means++)
2. **Assignment step**: assign each point to the nearest centroid
3. **Update step**: recompute each centroid as the mean of its assigned points
4. Repeat steps 2–3 until convergence (centroids stop moving)

K-Means is guaranteed to converge but may converge to a **local minimum**, so multiple restarts are recommended.

### Choosing K: The Elbow Method

Plot WCSS (inertia) vs. $K$ — look for the "elbow" where the curve bends, indicating diminishing returns from adding more clusters.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)
plt.rcParams['font.size'] = 12

## 2. K-Means from Scratch

In [ ]:
class KMeansScratch:
    """K-Means clustering using Lloyd's algorithm."""

    def __init__(self, k=3, max_iters=300, tol=1e-4, n_init=10):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol
        self.n_init = n_init

    def _init_centroids(self, X):
        """K-Means++ initialization for better starting positions."""
        idx = np.random.choice(len(X))
        centroids = [X[idx]]

        for _ in range(self.k - 1):
            dists = np.min([np.linalg.norm(X - c, axis=1)**2 for c in centroids], axis=0)
            probs = dists / dists.sum()
            idx = np.random.choice(len(X), p=probs)
            centroids.append(X[idx])

        return np.array(centroids)

    def _assign(self, X, centroids):
        dists = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        return np.argmin(dists, axis=1)

    def _inertia(self, X, labels, centroids):
        return sum(np.sum((X[labels == k] - centroids[k])**2) for k in range(self.k))

    def _run_once(self, X):
        centroids = self._init_centroids(X)
        for _ in range(self.max_iters):
            labels = self._assign(X, centroids)
            new_centroids = np.array([
                X[labels == k].mean(axis=0) if (labels == k).sum() > 0 else centroids[k]
                for k in range(self.k)
            ])
            if np.linalg.norm(new_centroids - centroids) < self.tol:
                break
            centroids = new_centroids
        return labels, centroids, self._inertia(X, labels, centroids)

    def fit(self, X):
        best = None
        for _ in range(self.n_init):
            labels, centroids, inertia = self._run_once(X)
            if best is None or inertia < best[2]:
                best = (labels, centroids, inertia)
        self.labels_, self.cluster_centers_, self.inertia_ = best
        return self

## 3. Synthetic Blobs: Visualize Clustering

In [ ]:
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

km = KMeansScratch(k=4)
km.fit(X_blobs)

ari = adjusted_rand_score(y_true, km.labels_)
sil = silhouette_score(X_blobs, km.labels_)

colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# True labels
for cls, color in zip(range(4), colors):
    mask = y_true == cls
    axes[0].scatter(X_blobs[mask, 0], X_blobs[mask, 1], c=color, alpha=0.7, s=50)
axes[0].set_title('True Labels')

# K-Means labels
for cls, color in zip(range(4), colors):
    mask = km.labels_ == cls
    axes[1].scatter(X_blobs[mask, 0], X_blobs[mask, 1], c=color, alpha=0.7, s=50)
axes[1].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
                c='black', marker='X', s=200, zorder=10, label='Centroids')
axes[1].set_title(f'K-Means (K=4)\nARI={ari:.3f}, Silhouette={sil:.3f}')
axes[1].legend()

plt.suptitle('K-Means Clustering on Synthetic Blobs', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. The Elbow Method: Choosing K

In [ ]:
k_range = range(1, 11)
inertias = []
silhouettes = []

for k in k_range:
    km_k = KMeansScratch(k=k, n_init=5)
    km_k.fit(X_blobs)
    inertias.append(km_k.inertia_)
    if k > 1:
        silhouettes.append(silhouette_score(X_blobs, km_k.labels_))
    else:
        silhouettes.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(k_range, inertias, 'o-', color='steelblue', lw=2)
axes[0].axvline(4, color='tomato', linestyle='--', label='True K=4')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method'); axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(list(k_range)[1:], silhouettes[1:], 's-', color='tomato', lw=2)
axes[1].axvline(4, color='steelblue', linestyle='--', label='True K=4')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Scores'); axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Choosing the Right K', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. K-Means on Iris Dataset

K-Means is unsupervised — it doesn't use labels during training. We can compare discovered clusters to the true classes afterward.

In [ ]:
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Scale features
scaler = StandardScaler()
X_iris_s = scaler.fit_transform(X_iris)

km_iris = KMeansScratch(k=3, n_init=10)
km_iris.fit(X_iris_s)

ari  = adjusted_rand_score(y_iris, km_iris.labels_)
sil  = silhouette_score(X_iris_s, km_iris.labels_)

print(f"Adjusted Rand Index: {ari:.4f}  (1.0 = perfect match with true labels)")
print(f"Silhouette Score:    {sil:.4f}  (higher = better-defined clusters)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#e74c3c', '#2ecc71', '#3498db']

# True labels
for cls, color, name in zip([0,1,2], colors, iris.target_names):
    mask = y_iris == cls
    axes[0].scatter(X_iris[mask, 2], X_iris[mask, 3], c=color, label=name,
                    edgecolors='k', s=60, alpha=0.8)
axes[0].set_xlabel('Petal Length'); axes[0].set_ylabel('Petal Width')
axes[0].set_title('True Species Labels'); axes[0].legend()

# K-Means assignments
for cls, color in zip([0,1,2], colors):
    mask = km_iris.labels_ == cls
    axes[1].scatter(X_iris[mask, 2], X_iris[mask, 3], c=color,
                    edgecolors='k', s=60, alpha=0.8, label=f'Cluster {cls}')
c = scaler.inverse_transform(km_iris.cluster_centers_)
axes[1].scatter(c[:, 2], c[:, 3], c='black', marker='X', s=200, zorder=10, label='Centroids')
axes[1].set_xlabel('Petal Length'); axes[1].set_ylabel('Petal Width')
axes[1].set_title(f'K-Means Clusters (ARI={ari:.3f})')
axes[1].legend()

plt.suptitle('K-Means on Iris: Discovered vs. True Clusters', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

### Key Takeaways

- K-Means minimizes **within-cluster sum of squares** using an expectation-maximization style loop.
- **K-Means++** initialization dramatically improves convergence stability by choosing initial centroids that are far apart.
- The **elbow method** and **silhouette score** help choose $K$ when true labels are unavailable.
- On Iris, K-Means recovers the true clusters well (high ARI), especially for petal features.
- K-Means assumes **spherical, equal-variance** clusters — it struggles with elongated, unequal, or non-convex shapes.
- Always **standardize** features before K-Means — features with larger scales will dominate the distance computation.